# Model Języka — polskie nazwy miast (PyTorch)

W tym projekcie zajmiemy się technologią, która jest bardzo popularna i przydatna oraz jest jednym z głównych komponentów takich modeli jak ChatGPT - *Language Model* : [Wikipedia](https://en.wikipedia.org/wiki/Language_model)<br>

W dużym uproszczeniu staramy się wypredykować kolejne słowo w danej sekwencji. Pomyślcie o następującym zdaniu:<br><br>
***Mamy dzisiaj bardzo słoneczną ...***<br><br>
Zapewne zdecydowana większość z was zaproponowałaby kończące słowo $pogodę$, być może ktoś zaporponowałby słowo $atmosferę$, ale prawie na pewno nikt nie zaproponowałby słowa $krzesło$.<br>
Uporaszczając właśnie takie zadanie, taką predykcję, wykonują modele takie jak ChatGPT (oczywiście pozwalają one również zadać, która bardziej ogólnie nazywamy promptem). Zatem nasz model ML'owy ma za zadanie wyestymować warunkowy rozkład słów (czyli prawdopodobieństwa dla poszczegółnych słów, wszystkich słów w słowniku) pod warunkiem tej niedokończonej sekwencji:<br><br>
```
P('pogodę' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) = ?

P('atmosferę' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) = ?

P('krzesło' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) = ?
```

Gdyby ktoś poprosił mnie abym oszacował te warnkowe prawdopodobieństwa zapewne przypisałbym następujące wartości (uwaga: to mój subiektywny szacunek, proszę nie traktować tych liczb jako wyliczonych na podstawie jakiś danych, chciałem raczej przekazać różnice pomiędzy tymi prawdopodobieństwami)

```
P('pogodę' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) = 0.95

P('atmosferę' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) = 0.03

P('krzesło' | ['Mamy', 'dzisiaj', 'bardzo', 'słoneczną']) ~ 0
```

**Oczywiście gdybyśmy zsumowali te prawdopodobieństwa warunkowe po wszystkich słowach w słowniku to otrzymalibyśmy wartość 1!**<br><br>

Zwykle taki *Language Model* musi być uczony na ogromnych korpusach danych, liczących Terabajty (np. cała wikipedia), my aby zaprezentować podobny mechanizm w naszym projekcie "zejdziemy jedno oczko niżej" - **Zamiast predykować następne słowo w zdaniu będziemy predykować następną literę w wyrazie.** (pomyślcie chociażby ile jest możliwych wszystkich słów w słowniku a ile jest możliwych liter! Takie znaczące zmniejszenie pomoże nam zbudować mały działający model).<br><br>
Nasz *Language model* będzie dotyczył "języka" polskich nazw miejscowości. Za pomocą naszego modelu będziemy mogli generować nowe nazwy polskich miast!

Dane pochodzą z https://dane.gov.pl/pl/dataset/188,wykaz-urzedowych-nazw-miejscowosci-i-ich-czesci

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

df_cities = pd.read_table("PRNG_MIEJSCOWOSCI_05_2021.csv", sep=';')
df_cities

In [ ]:
df_cities['Rodzaj miejscowości'].value_counts()

Do modelu wybieramy tylko nazwy miast. Proszę stworzyć listę, która będzie zawierała nazwy miast, filtrując tę informację na podstawie kolumny `Rodzaj miejscowości`

In [ ]:
list_of_cities = df_cities[df_cities['Rodzaj miejscowości'] == 'miasto']['Główna nazwa miejscowości'].to_list()

list_of_cities[:10]

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
#rozwiązanie
list_of_cities = df_cities[df_cities['Rodzaj miejscowości'] == 'miasto']['Główna nazwa miejscowości'].tolist()

list_of_cities[:10]

<h5>&nbsp;</h5>
<span style="font-family:Monospace"> ['Aleksandrów Kujawski',<br>
 'Aleksandrów Łódzki',<br>
 'Alwernia',<br>
 'Andrychów',<br>
 'Annopol',<br>
 'Augustów',<br>
 'Babimost',<br>
 'Baborów',<br>
 'Baranów Sandomierski',<br>
 'Barcin'] </span>

Każdą miejscowośc poprzedzamy tokenem sos (start of sentence) i kończymy eos (end of sentence):<br>
```
<sos>Wrocław</eos> 
```
W naszym przypadku umawiamy sie na % i ! - ponieważ model będzie oparty o pojedyncze litery i w takiej sytuacji jest wygodniej gdy specjalne tokeny również są pojedynczymi znakami.<br>
```
%Wrocław!
```
Proszę dodać w naszej liście do każdej nazwy miejscowości na początku symbol `%` a na końcu `!` (polecam użyć list comprehension):

In [ ]:
city_list = ['%' + city + '!' for city in list_of_cities]

city_list[:10]

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
#rozwiązanie
city_list = ['%' + city + '!' for city in list_of_cities]

city_list[:10]

<h5>&nbsp;</h5>
<span style="font-family:Monospace"> ['%Aleksandrów Kujawski!',<br>
 '%Aleksandrów Łódzki!',<br>
 '%Alwernia!',<br>
 '%Andrychów!',<br>
 '%Annopol!',<br>
 '%Augustów!',<br>
 '%Babimost!',<br>
 '%Baborów!',<br>
 '%Baranów Sandomierski!',<br>
 '%Barcin!']</span>

No dobrze - w większości mamy już gotowy zestaw danych, nasz "język" miast polskich. Teraz musimy pomyśleć nad modelem, który będzie estymował rozkład warunkowy poszczególnych liter/znaków. Do tego celu wykorzystamy sieci RNN.

## Tokenizacja znaków

W wersji TensorFlow korzystaliśmy z `Tokenizer(char_level=True)` z pakietu Keras. W PyTorch **nie ma wbudowanego odpowiednika** — napiszemy więc własną prostą klasę `CharTokenizer`, która:
- przypisuje każdemu unikalnemu znakowi indeks (zaczynając od 1, bo 0 rezerwujemy na padding),
- pozwala zamienić tekst na sekwencję indeksów i odwrotnie.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from matplotlib import pyplot as plt

# ---------- Wybór urządzenia (MPS na Apple Silicon / CUDA na GPU / CPU) ----------
device = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f"Urządzenie: {device}")

In [ ]:
class CharTokenizer:
    """Prosty tokenizer znakowy — odpowiednik Keras Tokenizer(char_level=True).
    
    Indeks 0 jest zarezerwowany na padding (nigdy nie jest przypisany do żadnego znaku).
    Znaki są sortowane malejąco po częstości (najczęstszy znak dostaje indeks 1),
    dzięki czemu kolejność jest identyczna jak w oryginalnym Tokenizerze Keras.
    """

    def __init__(self):
        self.word_index = {}   # char -> idx  (indeksy od 1)
        self.index_word = {}   # idx  -> char

    def fit_on_texts(self, texts):
        """Buduje słownik na podstawie listy tekstów."""
        from collections import Counter
        counter = Counter()
        for text in texts:
            counter.update(text)
        # sortujemy malejąco po częstości (jak Keras Tokenizer)
        for idx, (char, _) in enumerate(counter.most_common(), start=1):
            self.word_index[char] = idx
            self.index_word[idx] = char

    def texts_to_sequences(self, texts):
        """Zamienia listę tekstów na listę list indeksów."""
        return [[self.word_index[ch] for ch in text if ch in self.word_index] for text in texts]

    @property
    def vocab_size(self):
        """Rozmiar słownika łącznie z paddingiem (indeks 0)."""
        return len(self.word_index) + 1

In [ ]:
tokenizer = CharTokenizer()
tokenizer.fit_on_texts(city_list)

vocab_size = tokenizer.vocab_size
print(f'Wielkość naszego słownika wynosi: {vocab_size}')
# tokenizer.word_index

Możemy zobaczyć kilka elementów z naszego słownika.<br>
Można też zobaczyć całość wywołując poniższą komendę ale wynik będzie dość długi:
>tokenizer.word_index

In [ ]:
Nth = ['Pierwszy','Drugi','Trzeci', 'Czwarty', 'Piąty', 'Szósty', 'Siódmy']

for nth, items in zip(Nth, tokenizer.word_index.items()):
    print(f'{nth:8} - {items[0]}:{items[1]}')

Teraz czeka nast odpowiednie przygotowanie danych X i Y (objeśniających i objaśnianych) jako wejść do naszego modelu. Spróbujcie przełożyć nasz przykład, mając zdanie:
```
Mamy dzisiaj bardzo słoneczną pogodę
```
Przekształcamy je na:
```
              X                   ->   Y
Mamy dzisiaj bardzo słoneczną ... -> pogodę
```
Bardzo podobnie chcemy zrobić z naszymi nazwami miast:
```
Oryginalne wejście: %Baborów!

W jednej iteracji:
    X    -> Y
%Baborów -> !
P(! | %Baborów)
```
ale także, w jednej iteracji:
```
    X    -> Y
 %Baboró -> w
 P(w | %Baboró)

      X  -> Y
    %Bab -> o
    P(o | %Bab)
```

Okazuje się, że tych wiele iteracji będziemy w stanie zamknąć w jednym przebiegu modelu RNN zatem ostatecznie nie potrzebujemy tych wszystkich kombinacji o różnej długości dla jednej nazwy miasta ale następującego X i Y:
```
    X    ->    Y
%Baborów -> Baborów!
```
Model RNN będzie tak 'sprytny', że na podstawie takiego X i Y będzie liczył po kolei:
```
1. P(B | %)
2. P(a | %B)
3. P(b | %Ba)
...
```

Jeszcze jednym zagadnieniem technicznym jest normalizacja długości wejścia. Chcemy aby do sieci wchodziła macierz gdzie w pierwszej linii jest zakodowana pierwsza miejscowość (podzielona na poszczególne litery), w drugiej druga miejscowość itd. Problemem jest fakt, że miejscowości mają różne długości nazw. Zatem po prostu wyznaczymy najdłuższą nazwę miejscowości i dla wszystkich nazw krótszych niż ona będziemy dodawać same zera na końcu.<br><br> Powoduje to jednak pewien problem - krótsze nazwy będą składały się w dużej mierze z zer (paddingu). W takiej sytuacji model będzie uczył się że pad trzeba zamienić na pad. Nie jest to zła wiedza natomiast problematyczna z punktu widzenia procesu uczenia się sieci neuronowej - zamiana pad na pad jest bardzo łatwa i nie jest nam do niczego potrzebna. Jeżeli tokeny pad będą uwzględniane przy liczeniu loss to będzie ona sztucznie zaniżona. Dlatego w późniejszej części ustawiamy maskowanie paddingu.

> **Różnica PyTorch vs Keras:**<br>
> W Keras używaliśmy `mask_zero=True` w warstwie `Embedding` — maska propagowała się automatycznie przez całą sieć.<br>
> W PyTorch korzystamy z dwóch mechanizmów:
> - `padding_idx=0` w `nn.Embedding` — wektor embeddingu dla indeksu 0 jest zawsze zerowy i nie jest aktualizowany,
> - `ignore_index=0` w `nn.CrossEntropyLoss` — pozycje paddingu są pomijane przy obliczaniu straty.

Poniżej przykład jak zmienią się nazwy po paddingu:

```
%Babimost!000000000000
%Baranów Sandomierski!

%Baranów Sandomierski
Baranów Sandomierski!

%Babimost000000000000
Babimost!000000000000
```

In [ ]:
def pad_sequences(sequences, padding_value=0):
    """Padduje listę sekwencji (list intów) do wspólnej długości i zwraca numpy array."""
    max_len = max(len(seq) for seq in sequences)
    padded = np.full((len(sequences), max_len), fill_value=padding_value, dtype=np.int64)
    for i, seq in enumerate(sequences):
        padded[i, :len(seq)] = seq
    return padded

X_train_list = [city[:-1] for city in city_list]  # input: %Wrocław
y_train_list = [city[1:] for city in city_list]    # output: Wrocław!

print(f'example input: {X_train_list[6]}, example output: {y_train_list[6]}')

X_train_seq = tokenizer.texts_to_sequences(X_train_list)
y_train_seq = tokenizer.texts_to_sequences(y_train_list)

X_train_np = pad_sequences(X_train_seq)
y_train_np = pad_sequences(y_train_seq)

print(f'example input: {X_train_np[6]}, example output: {y_train_np[6]}')

In [ ]:
print(f'Kształt danych treningowych: {X_train_np.shape}')

# Konwersja na tensory PyTorch
X_train_t = torch.tensor(X_train_np, dtype=torch.long).to(device)
y_train_t = torch.tensor(y_train_np, dtype=torch.long).to(device)

print(f'X_train tensor: {X_train_t.shape}, device: {X_train_t.device}')
print(f'y_train tensor: {y_train_t.shape}, device: {y_train_t.device}')

## Budowa modelu

Dochodzimy do konstruowania modelu. Proszę uzupełnić funkcję `__init__` dla klasy `LanguageModel` zgodnie z podanymi poleceniami.

> **Różnica PyTorch vs Keras:**
> - W Keras warstwa `Dense(vocab_size, activation='softmax')` zwracała prawdopodobieństwa.
> - W PyTorch warstwa `nn.Linear(hidden_dim, vocab_size)` zwraca **surowe logity** (bez softmax). Softmax stosujemy **tylko podczas inferencji** (generowania). Podczas treningu `nn.CrossEntropyLoss` sam wewnętrznie stosuje log-softmax, więc podawanie mu logitów jest poprawne i numerycznie stabilniejsze.
> - W PyTorch nie ma `mask_zero=True` — zamiast tego używamy `padding_idx=0` w `nn.Embedding`.

In [ ]:
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=64):
        
        # użycie wszystkich elementów inicjalizacji klasy po której dziedziczy nasza klasa, czyli po nn.Module:
        super().__init__()
        
        ### Proszę poniżej uzupełnić kod ###
        
        # proszę zainicjalizować warstwę Embedding z pakietu PyTorch, z odpowiednim kształtem wejścia (ile "obiektów"
        # musimy obsłużyć?), wielkością wektora kodującego embed_dim, i flagą padding_idx=0 (ignorowanie paddingu)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # proszę zainicjalizować warstwę GRU (z rodziny modeli RNN) z pakietu PyTorch, z liczbą wejść embed_dim,
        # liczbą jednostek hidden_dim, parametrem batch_first=True (aby format danych to [batch, seq, features])
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        
        # proszę zainicjalizować gęstą warstwę (Linear) z pakietu PyTorch z odpowiednią liczbą wejść i wyjść -
        # dla ilu "obiektów" musimy wyznaczyć prawdopodobieństwa warunkowe?
        # UWAGA: w PyTorch NIE dodajemy softmax — Linear zwraca surowe logity!
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
        ### Koniec uzupełniania kodu ###
    
    def forward(self, x, hidden=None):
        """
        x      - wejście w formacie [batch_size, seq_len]
        hidden - initial state sieci RNN od którego sieć ma zacząć obliczenia. Jeżeli None — domyślny start (zera)
        
        Zwraca: (logits, hidden_state)
            logits       - surowe wyjścia [batch_size, seq_len, vocab_size]
            hidden_state - ostatni stan ukryty GRU [1, batch_size, hidden_dim]
        """
        # wyznaczamy embeddingi czyli wektorowe reprezentacje poszczególnych znaków
        emb = self.embedding(x)
        
        # zwróćcie proszę uwagę, że z GRU otrzymujemy 2 zmienne - output (sekwencja) i hidden (ostatni stan)
        # W PyTorch GRU z batch_first=True:
        #   output: [batch, seq_len, hidden_dim]
        #   hidden: [num_layers, batch, hidden_dim]
        output, hidden = self.gru(emb, hidden)
        
        # predykujemy rozkład warunkowy — surowe logity (bez softmax!)
        logits = self.fc(output)
        
        return logits, hidden

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
#rozwiązanie
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        """
        x      - wejście [batch_size, seq_len]
        hidden - stan ukryty [1, batch_size, hidden_dim] lub None
        
        Zwraca: (logits, hidden_state)
        """
        emb = self.embedding(x)
        output, hidden = self.gru(emb, hidden)
        logits = self.fc(output)
        return logits, hidden

model = LanguageModel(vocab_size).to(device)
print(model)
print(f'\nLiczba parametrów: {sum(p.numel() for p in model.parameters()):,}')

## Trening modelu

W PyTorch (w odróżnieniu od Keras) nie ma wbudowanego `model.compile()` + `model.fit()`. Musimy sami napisać pętlę treningową. Składa się ona z kilku kroków powtarzanych w każdej epoce:
1. **Forward pass** — przepuszczamy dane przez model i otrzymujemy logity
2. **Obliczenie straty** — porównujemy logity z oczekiwanymi etykietami
3. **Backward pass** — obliczamy gradienty (`loss.backward()`)
4. **Aktualizacja wag** — optimizer wykonuje krok (`optimizer.step()`)
5. **Zerowanie gradientów** — przed kolejną iteracją (`optimizer.zero_grad()`)

> **Ważne:** `nn.CrossEntropyLoss` oczekuje logitów w formacie `(N, C)` lub `(N, C, T)` gdzie C to liczba klas.
> Nasze logity mają kształt `(batch, seq_len, vocab_size)`, a targety `(batch, seq_len)`.
> Musimy więc zamienić osie logitów: `logits.permute(0, 2, 1)` aby uzyskać `(batch, vocab_size, seq_len)`.

Proszę uzupełnić poniższą komórkę:

In [ ]:
# proszę zainicjalizować błąd potrzebny do treningu modelu — CrossEntropyLoss z ignore_index=0
# (ignore_index=0 powoduje, że pozycje paddingu nie wpływają na wartość straty)
criterion = None

# proszę zainicjalizować optimizer Adam z pakietu torch.optim, podając parametry modelu
optimizer = None

EPOCHS = 150
loss_history = []

# proszę napisać pętlę treningową:
# for epoch in range(EPOCHS):
#     model.train()
#     optimizer.zero_grad()
#     logits, _ = model(X_train_t)
#     # WAŻNE: CrossEntropyLoss oczekuje (N, C, T) — permutujemy logity
#     loss = criterion(logits.permute(0, 2, 1), y_train_t)
#     loss.backward()
#     optimizer.step()
#     loss_history.append(loss.item())
#     if (epoch + 1) % 10 == 0:
#         print(f'Epoch {epoch+1:3d}/{EPOCHS} — loss: {loss.item():.4f}')

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
#rozwiązanie
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters())

EPOCHS = 150
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    
    logits, _ = model(X_train_t)
    
    # CrossEntropyLoss oczekuje (N, C, T) — permutujemy logity z (N, T, C) na (N, C, T)
    loss = criterion(logits.permute(0, 2, 1), y_train_t)
    
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} — loss: {loss.item():.4f}')

##### Wyrysujmy jak zachowywał się błąd w trakcie uczenia - czyli poprzez epoki

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(loss_history, label='Błąd uczenia', color='blue')
plt.xlabel('Epoki')
plt.ylabel('Błąd')
plt.title('Błąd uczenia w trakcie epok')
plt.legend()
plt.grid(True)
plt.show()

## Generowanie nazw miast (sampling)

<h5>&nbsp;</h5>
Po wytrenowaniu modelu jedyne co nam zostaje to sprawdzić jak działa! Proszę zwrócić uwagę, że jest to model generatywny czyli potrafi generować "nowe" obiekty - w naszej sytuacji nowe nazwy miejscowości.<br><br>
Chciałbym od razu uprzedzić, że niekoniecznie każdy wynik będzie brzmiał jak poprawna nazwa nieistniejącej polskiej miejscowości - czasem będzie niezrozumiałym szumem. Jednak jeśli faza treningu przebiegła poprawnie to przynajmniej część wygenerowanych nazw powinna brzmieć poprawnie.<br><br>
Proszę też zwrócić uwagę jak nietrywialnym jest zagadnienie ewaluacji i pomiaru jakości takiego modelu - proste metryki jak dokładność, F1 itp. nie mają tutaj niestety zastosowania.<br><br>
Podczas generowania sekwencji znaków możemy stosować różne metody próbkowania. Najprostszym podejściem jest przewidywanie rozkładu prawdopodobieństwa dla kolejnego znaku i wybór:<br>

- znaku o najwyższym prawdopodobieństwie (greedy search), co prowadzi do deterministycznych wyników,<br>
- znaku losowo, zgodnie z rozkładem prawdopodobieństwa (probabilistic sampling), co pozwala na bardziej różnorodne wyniki.<br><br>
Po ustaleniu kolejnego znaku dodajemy go do naszego prompt'u i ponawiamy całą procedurę. Powtarzamy ten krok albo 30 razy albo jeśli dojedziemy do naszego znaku końca nazwy miejscowości: `!` (model niejako sam "zrozumie", że w tym miejscu nazwa się kończy i nie powinien kontynuować).<br><br>
*Informacyjnie - poza `Greedy search` i `Probabilistic sampling` jest jeszcze `Beam search` - do implementacji dla chętnych.*

In [ ]:
# próbkowanie/inferencja modelu — wersja prosta (cały prompt przy każdym kroku)

def sample_model(prompt="%", greedy=False, print_outcome=True):
    model.eval()
    with torch.no_grad():
        # jeżeli po 30 krokach model nie uzna za stosowne wypredykować tokena </eos>, przerywamy
        for i in range(30):
            # enkodowanie promptu: zamiana znaków na indeksy zgodnie z tokenizerem
            encoded_prompt = tokenizer.texts_to_sequences([prompt])
            # zamiana na tensor PyTorch i przeniesienie na urządzenie
            input_tensor = torch.tensor(encoded_prompt, dtype=torch.long).to(device)
            
            # generujemy predykcję — model zwraca logity
            logits, _ = model(input_tensor)
            
            # interesuje nas rozkład po ostatnim znaku promptu
            # stosujemy softmax aby zamienić logity na prawdopodobieństwa
            last_logits = logits[0, -1, :]
            probs = F.softmax(last_logits, dim=-1).cpu().numpy()
            
            if greedy:
                # greedy sampling — zawsze wybieramy najbardziej prawdopodobny znak
                next_char_idx = probs.argmax()
            else:
                # probabilistic sampling — losujemy zgodnie z rozkładem prawdopodobieństwa
                next_char_idx = np.random.choice(len(probs), p=probs)
            
            # zamieniamy indeks na konkretny znak
            next_char = tokenizer.index_word[next_char_idx]
            
            # dodajemy znak do promptu
            prompt += next_char
            
            # jeżeli model wyznaczył ! to kończymy
            if next_char == '!':
                break
    
    if print_outcome:
        display(Markdown(f"## 🏘️ **Wygenerowana nazwa miasta:** `{prompt[1:-1].title()}`"))
    return prompt


# ustalamy wartość początkową (seed/prompt/warmup text)
prompt = "%"  # tylko <sos> — niech model wygeneruje nazwę na dowolną literę
# można również podać pierwszą lub kilka pierwszych liter:
# prompt = '%Kamil'

sample_model(prompt)

### Zoptymalizowane próbkowanie z ukrytym stanem (hidden state)

Jednak wywoływanie modelu w każdym kroku z rosnącym promptem jest nieefektywne. Zamiast tego możemy wykorzystać właściwości sieci rekurencyjnych (RNN), które przechowują informację o poprzednich znakach w postaci stanu ukrytego (hidden state). Dzięki temu wystarczy podawać do modelu jedynie ostatni wygenerowany znak oraz stan ukryty z poprzedniego kroku. Taka optymalizacja przyspiesza generację i lepiej wykorzystuje zdolność modelu do zapamiętywania kontekstu.

Teraz zobaczmy, jak wygląda kod implementujący tę zoptymalizowaną wersję.

In [ ]:
def visualize_probabilities(probabilities, step, tokenizer, prompt, show=True):
    """Wizualizacja rozkładu prawdopodobieństwa warunkowego dla kolejnego znaku."""
    characters = [char for idx, char in tokenizer.index_word.items() if idx > 0]
    
    # pomijamy indeks 0 (padding) w wizualizacji
    probabilities = probabilities[1:]

    plt.figure(figsize=(10, 6))
    plt.bar(characters, probabilities, color='skyblue')
    plt.title(f'Conditional Probability Distribution after Step {step}, current outcome: {prompt}')
    plt.xlabel('Characters')
    plt.ylabel('Probability')
    plt.ylim(0, 1)
    plt.xticks(rotation=45)
    plt.grid(axis='y')
    if show:
        plt.show()

In [ ]:
# Zaawansowane próbkowanie — z przekazywaniem hidden state

def sample_model_optimal(prompt='%', greedy=False, print_outcome=True, visuals=False):
    model.eval()
    with torch.no_grad():
        # enkodowanie promptu: zamiana znaków na indeksy
        encoded_prompt = tokenizer.texts_to_sequences([prompt])
        input_tensor = torch.tensor(encoded_prompt, dtype=torch.long).to(device)
        
        # zadeklarujmy pierwszy stan ukryty jako domyślny (None = zera)
        hidden_state = None
        
        # podobnie jak wcześniej nie zezwalamy na więcej niż 30 znaków
        for i in range(30):
            
            # predykujemy za pomocą modelu logity oraz ostatni hidden state
            # przekazujemy również stan ukryty podczas wywoływania modelu
            logits, hidden_state = model(input_tensor, hidden=hidden_state)
            
            # interesuje nas rozkład po ostatnim znaku — stosujemy softmax na logitach
            last_logits = logits[0, -1, :]
            probs = F.softmax(last_logits, dim=-1).cpu().numpy()

            if visuals:
                visualize_probabilities(probs, i, tokenizer, prompt, show=False)
            
            if greedy:
                # greedy sampling
                next_char_idx = probs.argmax()
            else:
                # probabilistic sampling
                next_char_idx = np.random.choice(len(probs), p=probs)
                
            # zamieniamy indeks na konkretny znak
            next_char = tokenizer.index_word[next_char_idx]
            
            # dodajemy znak do promptu
            prompt += next_char

            # jednak inaczej niż poprzednio — kolejne wejście zawiera TYLKO ostatni wygenerowany znak
            encoded_next = tokenizer.texts_to_sequences([next_char])
            input_tensor = torch.tensor(encoded_next, dtype=torch.long).to(device)

            # jeżeli model wyznaczył ! to kończymy
            if next_char == '!':
                break

    if print_outcome:
        display(Markdown(f"## 🏘️ **Wygenerowana nazwa miasta:** `{prompt[1:-1].title()}`"))
    if visuals:
        plt.show()
    return prompt


# ustalamy wartość początkową
prompt = '%Adam'

print(sample_model_optimal(prompt, visuals=True, greedy=False))

### Wygenerujmy kilka nazw na raz

In [ ]:
# Generujemy 10 nazw miast — probabilistic sampling
for _ in range(10):
    sample_model_optimal('%', greedy=False)

In [ ]:
# Greedy sampling — zawsze ten sam wynik dla danego promptu
sample_model_optimal('%', greedy=True)
sample_model_optimal('%Wr', greedy=True)
sample_model_optimal('%Kamil', greedy=True)

### Kontynuacja

Polecam dalszą pracę z powyższym kodem. Kilka propozycji co można wykonać:
 - Zamiana hiperparametrów modelu tak aby błąd podczas uczenia był jeszcze mniejszy: hyperparameter tuning
 - Wyuczenie modelu języka dla nazw wsi, części miasta, historycznej nazwy miejscowości
 - Beam Search podczas inferencji